# Setup

In [1]:
# Optional autoreload while developing locally

%load_ext autoreload
%autoreload 2

In [2]:

# 0) Local-only setup and base diagnostics
import os
import sys
import platform
from pathlib import Path

ENV = "Local"

print("ENV:", ENV)
print("Python:", sys.version)
print("Platform:", platform.platform())
print("CUDA visible devices:", os.environ.get("CUDA_VISIBLE_DEVICES"))

required_files = {
    "tokenizer.py",
    "doc_chunker.py",
    "coreference_cluster_extractor.py",
    "local_clusters.py",
    "artifact_io.py",
}

PROJECT_DIR = None
local_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]

for candidate in local_candidates:
    if candidate.exists() and required_files.issubset(
        {x.name for x in candidate.iterdir() if x.is_file()}
    ):
        PROJECT_DIR = candidate
        break

if PROJECT_DIR is None:
    PROJECT_DIR = Path.cwd()
    print(
        "[warning] Could not find all project .py files in cwd/parents. "
        "Using current working directory anyway:", PROJECT_DIR
    )

sys.path.insert(0, str(PROJECT_DIR))
print("PROJECT_DIR =", PROJECT_DIR)
print("sys.path[0] =", sys.path[0])
print("Project files found:")
for name in sorted(required_files):
    path = PROJECT_DIR / name
    print(" -", name, "OK" if path.exists() else "MISSING")


ENV: Local
Python: 3.10.4 (tags/v3.10.4:9d38120, Mar 23 2022, 23:13:41) [MSC v.1929 64 bit (AMD64)]
Platform: Windows-10-10.0.22631-SP0
CUDA visible devices: None
PROJECT_DIR = c:\Università\Magistrale\NLP - Natural Language Processing\00 Progetto Finale\src_MAVERICK_SpaCyTagger\WORKING DIR
sys.path[0] = c:\Università\Magistrale\NLP - Natural Language Processing\00 Progetto Finale\src_MAVERICK_SpaCyTagger\WORKING DIR
Project files found:
 - artifact_io.py OK
 - coreference_cluster_extractor.py OK
 - doc_chunker.py OK
 - local_clusters.py OK
 - tokenizer.py OK


In [3]:

# 0.3) Shared device diagnostics
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    print("GPU 0:", torch.cuda.get_device_name(0))
    DEVICE = "cuda:0"
else:
    DEVICE = "cpu"
print("Using DEVICE =", DEVICE)


torch: 2.6.0+cu118
CUDA available: True
GPU count: 1
GPU 0: NVIDIA GeForce GTX 1050 with Max-Q Design
Using DEVICE = cuda:0


# Imports

In [4]:

import pickle
import re

import pandas as pd

from tokenizer import create_tokenizer, ensure_booknlp_extensions, tokens_to_dicts
from doc_chunker import create_chunker
from coreference_cluster_extractor import create_coreference_cluster_extractor
from local_clusters import (
    chunk_specs_to_rows,
    create_local_clusters_metadata,
    extracted_chunk_to_rows,
    validate_chunk_rows,
)
from artifact_io import LocalClusterMarathonStore, load_local_clusters_artifact

import torch
import torch.serialization as torch_serialization


In [5]:
# from global_coref_merger import merge_local_coreference_clusters # -> connected-components version
from global_coref_merger_aggregative import merge_local_coreference_clusters # -> aggregative version

from mention_filter import (build_anchor_bank, filter_mentions_by_foreign_anchors,load_generic_anchor_keys)

# Functions

In [6]:
# text loading + preprocessing
def load_txt(path):
    path = Path(path)

    try:
        return path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        return path.read_text(encoding="latin-1")

In [7]:
def preprocess_for_NER(text):
    # Normalize Windows/Mac newlines
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove possible invisible BOM
    text = text.replace("\ufeff", "")

    # Normalize curly apostrophes/quotes
    text = text.replace("’", "'")
    text = text.replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')

    # Replace line breaks inside paragraphs with spaces,
    # but preserve paragraph boundaries
    paragraphs = text.split("\n\n")
    paragraphs = [
        re.sub(r"\s*\n\s*", " ", paragraph).strip()
        for paragraph in paragraphs
        if paragraph.strip()
    ]

    text = "\n\n".join(paragraphs)

    chapter_re = re.compile(
        r"(?im)^\s*chapter\s+(?:[ivxlcdm]+|\d+)\b[^\n]*\n*"
    )

    text = chapter_re.sub("", text)

    # Normalize multiple spaces
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

In [8]:

def resolve_text_path(
    *,
    project_dir: Path,
    filename: str = "oz_full.txt",
    local_path: str | Path = "../../datasets/libri/oz_full.txt",
) -> Path:
    """Resolve the local input text path."""
    text_path = Path(local_path)
    if text_path.exists():
        return text_path

    fallback_path = project_dir / filename
    if fallback_path.exists():
        return fallback_path

    raise FileNotFoundError(
        "Could not resolve a local text file. "
        f"Tried LOCAL_TEXT_PATH={text_path} and PROJECT_DIR / {filename!r} = {fallback_path}."
    )


In [9]:

def save_doc(doc, path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb") as f:
        pickle.dump(doc, f)
    return path


def load_doc(path: str | Path):
    # Custom spaCy extensions must be registered before unpickling the Doc.
    ensure_booknlp_extensions()
    path = Path(path)
    with path.open("rb") as f:
        return pickle.load(f)


def detect_runtime_profile(*, force_profile: str | None = None) -> dict:
    """Return one explicit local runtime profile."""
    import torch

    cuda_available = torch.cuda.is_available()
    gpu_count = torch.cuda.device_count() if cuda_available else 0
    gpu_name = torch.cuda.get_device_name(0) if cuda_available else ""

    if force_profile:
        kind = force_profile
    elif cuda_available:
        kind = "local_cuda"
    else:
        kind = "local_cpu"

    defaults = {
        "local_cpu": {"chunk_size": 3000, "overlap_sentences": 30},
        "local_cuda": {"chunk_size": 6000, "overlap_sentences": 60},
    }
    profile_defaults = defaults.get(kind, defaults["local_cpu"])

    return {
        "kind": kind,
        "env": "Local",
        "cuda_available": cuda_available,
        "gpu_count": gpu_count,
        "gpu_name": gpu_name,
        "device": "cuda:0" if cuda_available else "cpu",
        "cpu_load_first": bool(cuda_available),
        "precision_policy": "auto",
        "p100_fallback_to_float32": False,
        "default_chunk_size": profile_defaults["chunk_size"],
        "default_overlap_sentences": profile_defaults["overlap_sentences"],
    }


def memory_snapshot() -> dict:
    snapshot = {}
    try:
        import resource
        snapshot["max_rss_mb"] = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024
    except Exception:
        pass
    try:
        import torch
        if torch.cuda.is_available():
            snapshot["cuda_allocated_mb"] = torch.cuda.memory_allocated() / (1024 ** 2)
            snapshot["cuda_reserved_mb"] = torch.cuda.memory_reserved() / (1024 ** 2)
    except Exception:
        pass
    return snapshot


def cleanup_after_chunk(runtime_profile: dict) -> None:
    import gc

    gc.collect()
    if str(runtime_profile.get("device", "")).startswith("cuda"):
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                if hasattr(torch.cuda, "ipc_collect"):
                    torch.cuda.ipc_collect()
        except Exception:
            pass


In [10]:
def patch_torch_load_weights_only_false() -> None:
    """
    Idempotently patch torch.load so PyTorch/Lightning checkpoint loading uses
    weights_only=False, without creating recursive wrappers when the cell is
    executed multiple times.
    """

    # Recover the real PyTorch implementation if torch.load was already patched.
    # In normal PyTorch, torch.load is exported from torch.serialization.load.
    torch.load = torch_serialization.load

    if not hasattr(torch, "_maverick_original_torch_load"):
        torch._maverick_original_torch_load = torch_serialization.load

    def _torch_load_force_weights_only_false(*args, **kwargs):
        kwargs["weights_only"] = False
        return torch._maverick_original_torch_load(*args, **kwargs)

    _torch_load_force_weights_only_false._maverick_weights_patch = True
    torch.load = _torch_load_force_weights_only_false

# Config

## I/O config

In [11]:

# Requested local file
TEXT_FILENAME = "oz_full.txt"
LOCAL_TEXT_PATH = Path(f"../../datasets/libri/{TEXT_FILENAME}")

TEXT_PATH = resolve_text_path(
    project_dir=PROJECT_DIR,
    filename=TEXT_FILENAME,
    local_path=LOCAL_TEXT_PATH,
)

OUTPUT_ROOT = PROJECT_DIR / "outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

TOKENIZED_DOC_PATH = OUTPUT_ROOT / "tokenized_doc.pkl"
LOCAL_CLUSTERS_ARTIFACT_PATH = OUTPUT_ROOT / "maverick_local_clusters.zip"
LOCAL_MARATHON_CACHE_DIR = OUTPUT_ROOT / ".maverick_local_cache"

print("TEXT_PATH =", TEXT_PATH)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("TOKENIZED_DOC_PATH =", TOKENIZED_DOC_PATH)
print("LOCAL_CLUSTERS_ARTIFACT_PATH =", LOCAL_CLUSTERS_ARTIFACT_PATH)
print("LOCAL_MARATHON_CACHE_DIR =", LOCAL_MARATHON_CACHE_DIR)


TEXT_PATH = ..\..\datasets\libri\oz_full.txt
OUTPUT_ROOT = c:\Università\Magistrale\NLP - Natural Language Processing\00 Progetto Finale\src_MAVERICK_SpaCyTagger\WORKING DIR\outputs
TOKENIZED_DOC_PATH = c:\Università\Magistrale\NLP - Natural Language Processing\00 Progetto Finale\src_MAVERICK_SpaCyTagger\WORKING DIR\outputs\tokenized_doc.pkl
LOCAL_CLUSTERS_ARTIFACT_PATH = c:\Università\Magistrale\NLP - Natural Language Processing\00 Progetto Finale\src_MAVERICK_SpaCyTagger\WORKING DIR\outputs\maverick_local_clusters.zip
LOCAL_MARATHON_CACHE_DIR = c:\Università\Magistrale\NLP - Natural Language Processing\00 Progetto Finale\src_MAVERICK_SpaCyTagger\WORKING DIR\outputs\.maverick_local_cache


In [12]:
GLOBAL_COREF_OUTPUT_DIR = OUTPUT_ROOT / "global_coref"
GLOBAL_COREF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("GLOBAL_COREF_OUTPUT_DIR =", GLOBAL_COREF_OUTPUT_DIR)

GLOBAL_COREF_OUTPUT_DIR = c:\Università\Magistrale\NLP - Natural Language Processing\00 Progetto Finale\src_MAVERICK_SpaCyTagger\WORKING DIR\outputs\global_coref


## Runtime and pipeline config

In [13]:

# Pipeline configuration
SPACY_MODEL = "en_core_web_sm"
TOKENIZER_DISABLE = ("ner",)

HF_NAME_OR_PATH = "sapienzanlp/maverick-mes-litbank"
SINGLETONS = True
CLEAN_MENTIONS = True
MAX_MENTION_TOKENS = 20
DROP_CROSS_SENTENCE_MENTIONS = True

# Local runtime profile. Set FORCE_RUNTIME_PROFILE to None, "local_cpu", or "local_cuda".
FORCE_RUNTIME_PROFILE = None
RUNTIME_PROFILE = detect_runtime_profile(force_profile=FORCE_RUNTIME_PROFILE)
DEVICE = RUNTIME_PROFILE["device"]

# CHUNK_SIZE means expanded chunk size, including overlap.
# If CHUNK_SIZE = 2000, every materialized DocChunk must contain <= 2000 tokens total.
CHUNK_SIZE = RUNTIME_PROFILE["default_chunk_size"]
OVERLAP_SENTENCES = RUNTIME_PROFILE["default_overlap_sentences"]
MAX_EXPANDED_CHUNK_TOKENS = CHUNK_SIZE

# Local marathon-cache behavior.
LOCAL_CLUSTER_EXECUTION_MODE = "marathon_cache"
FAIL_ON_INCOMPATIBLE_MANIFEST = True
RECOMPUTE_CORRUPT_CHUNKS = True
KEEP_CORRUPT_SHARD_BACKUPS = True
FORCE_RECOMPUTE_CHUNKS = set()
FORCE_RECOMPUTE_ALL_LOCAL_CLUSTERS = False
ENABLE_SHARD_CHECKSUMS = True
FINAL_STREAMING_VALIDATE_GLOBAL_IDS = True
DELETE_CACHE_AFTER_FINALIZE = False
OVERWRITE_EXISTING_FINAL_ZIP = True

# Per-chunk validation catches broken rows before committing a shard.
PER_CHUNK_VALIDATION_MODE = "basic"  # "basic" or "debug"

print("RUNTIME_PROFILE =", RUNTIME_PROFILE)
print("DEVICE =", DEVICE)
print("CHUNK_SIZE(expanded) =", CHUNK_SIZE)
print("OVERLAP_SENTENCES =", OVERLAP_SENTENCES)
print("MAX_EXPANDED_CHUNK_TOKENS =", MAX_EXPANDED_CHUNK_TOKENS)
print("LOCAL_CLUSTER_EXECUTION_MODE =", LOCAL_CLUSTER_EXECUTION_MODE)
print("LOCAL_MARATHON_CACHE_DIR =", LOCAL_MARATHON_CACHE_DIR)
print("PER_CHUNK_VALIDATION_MODE =", PER_CHUNK_VALIDATION_MODE)


RUNTIME_PROFILE = {'kind': 'local_cuda', 'env': 'Local', 'cuda_available': True, 'gpu_count': 1, 'gpu_name': 'NVIDIA GeForce GTX 1050 with Max-Q Design', 'device': 'cuda:0', 'cpu_load_first': True, 'precision_policy': 'auto', 'p100_fallback_to_float32': False, 'default_chunk_size': 6000, 'default_overlap_sentences': 60}
DEVICE = cuda:0
CHUNK_SIZE(expanded) = 6000
OVERLAP_SENTENCES = 60
MAX_EXPANDED_CHUNK_TOKENS = 6000
LOCAL_CLUSTER_EXECUTION_MODE = marathon_cache
LOCAL_MARATHON_CACHE_DIR = c:\Università\Magistrale\NLP - Natural Language Processing\00 Progetto Finale\src_MAVERICK_SpaCyTagger\WORKING DIR\outputs\.maverick_local_cache
PER_CHUNK_VALIDATION_MODE = basic


# Main

### Tokenization

In [14]:
# Load or create the stable tokenized Doc artifact.
if TOKENIZED_DOC_PATH.exists():
    print(f"[doc] Loading tokenized Doc from {TOKENIZED_DOC_PATH}")
    doc = load_doc(TOKENIZED_DOC_PATH)
else:
    print("[doc] Tokenized Doc not found; creating it from raw text.")
    raw_text = load_txt(TEXT_PATH)
    text = preprocess_for_NER(raw_text)
    tokenizer = create_tokenizer(
        SPACY_MODEL,
        disable=TOKENIZER_DISABLE,
        verbose=True,
    )
    doc = tokenizer.tokenize(text)
    save_doc(doc, TOKENIZED_DOC_PATH)
    print(f"[doc] Saved tokenized Doc to {TOKENIZED_DOC_PATH}")

print("Doc tokens:", len(doc))
print("Doc sentences:", sum(1 for _ in doc.sents))

[doc] Loading tokenized Doc from c:\Università\Magistrale\NLP - Natural Language Processing\00 Progetto Finale\src_MAVERICK_SpaCyTagger\WORKING DIR\outputs\tokenized_doc.pkl
Doc tokens: 48045
Doc sentences: 1943


In [15]:
# df_tokens = pd.DataFrame(tokens_to_dicts(doc))
# df_tokens.head()

### Chunking

In [16]:
chunker = create_chunker(
    chunk_size=CHUNK_SIZE,
    overlap_sentences=OVERLAP_SENTENCES,
    max_expanded_chunk_tokens=MAX_EXPANDED_CHUNK_TOKENS,
)
chunk_plan = chunker.plan(doc)

In [17]:
# print(f"Planned {len(chunk_plan)} chunks without materializing chunk docs.")
# for spec in chunk_plan.specs[:5]:
#     print(
#         spec.chunk_id,
#         "expanded=", (spec.global_start, spec.global_end),
#         "expanded_tokens=", spec.n_tokens,
#         "core=", (spec.core_start, spec.core_end),
#         "left_overlap=", (spec.left_overlap_start, spec.left_overlap_end),
#         "right_overlap=", (spec.right_overlap_start, spec.right_overlap_end),
#     )


### Coreference clusters extraction

##### Local (chunk-level) coreference cluster extraction

In [18]:
patch_torch_load_weights_only_false() # Recover and safely patch torch.load for Maverick / PyTorch checkpoint loading
print("torch.load safely patched:", getattr(torch.load, "_maverick_weights_patch", False))

torch.load safely patched: True


In [19]:
# 2) Maverick local extraction: local marathon cache, per-chunk shards, streaming final aggregation.
coref_extractor = create_coreference_cluster_extractor(
    hf_name_or_path=HF_NAME_OR_PATH,
    device=DEVICE,
    singletons=SINGLETONS,
    verbose=True,
    clean_mentions=CLEAN_MENTIONS,
    max_mention_tokens=MAX_MENTION_TOKENS,
    drop_cross_sentence_mentions=DROP_CROSS_SENTENCE_MENTIONS,
    cpu_load_first=RUNTIME_PROFILE["cpu_load_first"],
    precision_policy=RUNTIME_PROFILE["precision_policy"],
    p100_fallback_to_float32=RUNTIME_PROFILE["p100_fallback_to_float32"],
)

maverick_config = {
    "hf_name_or_path": HF_NAME_OR_PATH,
    "device": DEVICE,
    "singletons": SINGLETONS,
    "clean_mentions": CLEAN_MENTIONS,
    "max_mention_tokens": MAX_MENTION_TOKENS,
    "drop_cross_sentence_mentions": DROP_CROSS_SENTENCE_MENTIONS,
    "chunk_size": CHUNK_SIZE,
    "chunk_size_semantics": "expanded_chunk_tokens_including_overlap",
    "overlap_sentences": OVERLAP_SENTENCES,
    "max_expanded_chunk_tokens": MAX_EXPANDED_CHUNK_TOKENS,
    "runtime_profile": RUNTIME_PROFILE,
}

metadata = create_local_clusters_metadata(
    doc=doc,
    chunk_count=len(chunk_plan),
    chunk_size=CHUNK_SIZE,
    overlap_sentences=OVERLAP_SENTENCES,
    max_expanded_chunk_tokens=MAX_EXPANDED_CHUNK_TOKENS,
    maverick_config=maverick_config,
    document_id=TEXT_PATH.stem,
)

store = LocalClusterMarathonStore(
    cache_dir=LOCAL_MARATHON_CACHE_DIR,
    output_zip_path=LOCAL_CLUSTERS_ARTIFACT_PATH,
    metadata=metadata,
    chunk_plan=chunk_plan,
    maverick_config=maverick_config,
    recompute_corrupt_chunks=RECOMPUTE_CORRUPT_CHUNKS,
    keep_corrupt_shard_backups=KEEP_CORRUPT_SHARD_BACKUPS,
    force_recompute_chunks=FORCE_RECOMPUTE_CHUNKS,
    force_recompute_all=FORCE_RECOMPUTE_ALL_LOCAL_CLUSTERS,
    enable_checksums=ENABLE_SHARD_CHECKSUMS,
    final_streaming_validate_global_ids=FINAL_STREAMING_VALIDATE_GLOBAL_IDS,
    overwrite_existing_final_zip=OVERWRITE_EXISTING_FINAL_ZIP,
    delete_cache_after_finalize=DELETE_CACHE_AFTER_FINALIZE,
)

store.validate_or_initialize_run()

current_chunk_id = None
try:
    for spec in chunk_plan.specs:
        current_chunk_id = spec.chunk_id

        if not store.should_process_chunk(spec):
            print(f"[chunk][skip] {spec.chunk_id}")
            continue

        print(
            f"[chunk] Processing {spec.chunk_id} "
            f"({spec.chunk_index + 1}/{len(chunk_plan)}) "
            f"expanded_tokens={spec.n_tokens}"
        )
        store.mark_chunk_started(spec)

        chunk = chunker.materialize(doc, spec)
        extracted = coref_extractor.extract(chunk)
        chunk_rows = extracted_chunk_to_rows(chunk=chunk, extracted=extracted)

        chunk_validation_report = validate_chunk_rows(
            doc=doc,
            chunk=chunk,
            rows=chunk_rows,
            mode=PER_CHUNK_VALIDATION_MODE,
        )

        store.commit_chunk(
            spec=spec,
            rows=chunk_rows,
            validation_report={
                **chunk_validation_report,
                "memory_after_validation": memory_snapshot(),
            },
        )

        print(
            f"[chunk] {spec.chunk_id}: "
            f"clusters={len(chunk_rows.clusters_rows)}, "
            f"mentions={len(chunk_rows.mentions_rows)}"
        )

        del chunk_rows
        del extracted
        del chunk
        coref_extractor.clear_runtime_state()
        cleanup_after_chunk(RUNTIME_PROFILE)

    store.validate_all_shards_complete()
    artifact_path = store.finalize_streaming()
    print(f"Saved final artifact to {artifact_path}")

except Exception as exc:
    if current_chunk_id is not None:
        # Find the current spec without relying on partially materialized runtime objects.
        failing_spec = next((s for s in chunk_plan.specs if s.chunk_id == current_chunk_id), None)
        if failing_spec is not None:
            store.record_chunk_failure(spec=failing_spec, exc=exc)
    print("[failure] The final artifact was not created successfully.")
    print("[failure] Marathon cache directory:", LOCAL_MARATHON_CACHE_DIR)
    raise
finally:
    try:
        coref_extractor.clear_runtime_state()
    except Exception:
        pass
    cleanup_after_chunk(RUNTIME_PROFILE)


[coref] Extractor build: streaming-safe device='cuda:0', precision_policy='auto', cpu_load_first=True.
[chunk][skip] chunk_000
[chunk][skip] chunk_001
[chunk][skip] chunk_002
[chunk][skip] chunk_003
[chunk][skip] chunk_004
[chunk][skip] chunk_005
[chunk][skip] chunk_006
[chunk][skip] chunk_007
[chunk][skip] chunk_008
[chunk][skip] chunk_009
[chunk][skip] chunk_010
[chunk][skip] chunk_011
[chunk][skip] chunk_012
[chunk][skip] chunk_013
[chunk][skip] chunk_014
[chunk][skip] chunk_015
Saved final artifact to c:\Università\Magistrale\NLP - Natural Language Processing\00 Progetto Finale\src_MAVERICK_SpaCyTagger\WORKING DIR\outputs\maverick_local_clusters.zip


##### local (chunk-level) coreference cluster artifact finalization

In [20]:

# 3) Optional artifact inspection without re-running Maverick
if not LOCAL_CLUSTERS_ARTIFACT_PATH.exists():
    raise FileNotFoundError(f"Final artifact does not exist yet: {LOCAL_CLUSTERS_ARTIFACT_PATH}")

loaded_tables = load_local_clusters_artifact(LOCAL_CLUSTERS_ARTIFACT_PATH)
print("Artifact metadata:")
print(loaded_tables.metadata)


Artifact metadata:
{'artifact_type': 'local_clusters', 'chunking': {'chunk_size': 6000, 'chunk_size_semantics': 'expanded_chunk_tokens_including_overlap', 'chunk_unit': 'sentence_aligned_spacy_tokens', 'max_expanded_chunk_tokens': 6000, 'n_chunks': 16, 'overlap_sentences': 60, 'sentence_aligned': True}, 'created_at': '2026-06-17T14:26:57.409305+00:00', 'document': {'document_id': 'oz_full', 'sentence_count': 1943, 'text_hash': 'sha256:a7f7353b34ce4ba9f0605e0e1a200304c3c679aa942dc22fcdb7a66d76aba8c1', 'token_count': 48045}, 'files': {'chunks.csv': {'rows': 16}, 'clusters.csv': {'rows': 1623}, 'mentions.csv': {'rows': 14633}}, 'marathon_cache': {'checksums_enabled': True, 'chunk_plan_fingerprint': 'sha256:2148dcb7bec000c1f289cf24d3197df7b1f0de78a541e86b29bb94924e201a66', 'final_streaming_validate_global_ids': True, 'maverick_config_fingerprint': 'sha256:e7a3daab1487232f64112d4947c2b0891ed93a9ef5375de09303ec0f372fff2b', 'private_cache_included_in_artifact': False, 'run_fingerprint': 'sha2

In [21]:
# Optional mention preview
pd.DataFrame(loaded_tables.clusters_rows)

,local_cluster_uid,chunk_id,chunk_index,local_cluster_id,canonical_name,n_mentions
0,chunk_000::cluster_000,chunk_000,0,0,the great Kansas prairies,1
1,chunk_000::cluster_001,chunk_000,0,1,a farmer,1
2,chunk_000::cluster_002,chunk_000,0,2,Aunt Em,118
3,chunk_000::cluster_003,chunk_000,0,3,the farmer,1
4,chunk_000::cluster_004,chunk_000,0,4,the house,35
...,...,...,...,...,...,...
1618,chunk_015::cluster_074,chunk_015,15,74,her friends,1
1619,chunk_015::cluster_075,chunk_015,15,75,the broad Kansas prairie,1
1620,chunk_015::cluster_076,chunk_015,15,76,the old one,1
1621,chunk_015::cluster_077,chunk_015,15,77,the barnyard,1


##### Destructive filter

In [22]:
from pathlib import Path
import sys
import pandas as pd

# If filter_mentions_foreign_anchor.py is in the same directory as the notebook,
# this is usually enough.
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))



# ---- Config ----------------------------------------------------------------

DESTRUCTIVE_FILTER_ENABLED = True

DESTRUCTIVE_FILTER_OUTPUT_DIR = GLOBAL_COREF_OUTPUT_DIR / "destructive_mention_filter"
DESTRUCTIVE_FILTER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILTERED_MENTIONS_PATH = DESTRUCTIVE_FILTER_OUTPUT_DIR / "mentions.filtered.csv"
DROPPED_MENTIONS_PATH = DESTRUCTIVE_FILTER_OUTPUT_DIR / "mentions.dropped.csv"
SELECTED_ANCHORS_PATH = DESTRUCTIVE_FILTER_OUTPUT_DIR / "selected_anchors.csv"

ANCHOR_PERCENTILE = 0.95          # top 5%
MIN_ANCHOR_MENTIONS = 2
FUZZY_THRESHOLD = 0.86
OWN_FAMILY_FUZZY_THRESHOLD = 0.90
MIN_FUZZY_CHARS = 5

# Optional custom blacklist file.
# Use None to rely only on the built-in conservative generic blacklist.
GENERIC_ANCHOR_BLACKLIST_PATH = None


# ---- Small adapters ---------------------------------------------------------

def rows_to_dataframe(rows) -> pd.DataFrame:
    """
    Accept either a DataFrame or a list[dict]-style row collection.
    """
    if isinstance(rows, pd.DataFrame):
        return rows.copy()
    return pd.DataFrame(rows)


def dataframe_to_original_row_shape(df: pd.DataFrame, original_rows):
    """
    Return DataFrame if the pipeline originally used DataFrames.
    Return list[dict] if the pipeline originally used row dictionaries.
    """
    if isinstance(original_rows, pd.DataFrame):
        return df
    return df.to_dict(orient="records")


# ---- Apply filter -----------------------------------------------------------

if DESTRUCTIVE_FILTER_ENABLED:
    clusters_df = rows_to_dataframe(loaded_tables.clusters_rows)
    mentions_df = rows_to_dataframe(loaded_tables.mentions_rows)

    print("[destructive-filter] Building foreign-anchor bank...")

    generic_anchor_keys = load_generic_anchor_keys(GENERIC_ANCHOR_BLACKLIST_PATH)

    anchors = build_anchor_bank(
        mentions=mentions_df,
        clusters=clusters_df,
        anchor_percentile=ANCHOR_PERCENTILE,
        min_anchor_mentions=MIN_ANCHOR_MENTIONS,
        generic_anchor_keys=generic_anchor_keys,
    )

    print(f"[destructive-filter] Selected anchors: {len(anchors):,}")

    filtered_mentions_df, dropped_mentions_df = filter_mentions_by_foreign_anchors(
        mentions=mentions_df,
        clusters=clusters_df,
        anchors=anchors,
        fuzzy_threshold=FUZZY_THRESHOLD,
        own_family_fuzzy_threshold=OWN_FAMILY_FUZZY_THRESHOLD,
        min_fuzzy_chars=MIN_FUZZY_CHARS,
    )

    selected_anchors_df = pd.DataFrame([
        {
            "anchor_key": anchor.key,
            "display_name": anchor.display_name,
            "n_mentions": anchor.n_mentions,
            "source_cluster_uids": "|".join(anchor.source_cluster_uids),
        }
        for anchor in anchors
    ])

    filtered_mentions_df.to_csv(FILTERED_MENTIONS_PATH, index=False)
    dropped_mentions_df.to_csv(DROPPED_MENTIONS_PATH, index=False)
    selected_anchors_df.to_csv(SELECTED_ANCHORS_PATH, index=False)

    before = len(mentions_df)
    after = len(filtered_mentions_df)
    dropped = before - after

    print("[destructive-filter] Done.")
    print(f"  Mentions before : {before:,}")
    print(f"  Mentions after  : {after:,}")
    print(f"  Dropped         : {dropped:,} ({(dropped / before * 100) if before else 0:.2f}%)")
    print(f"  Audit dropped   : {DROPPED_MENTIONS_PATH}")
    print(f"  Audit anchors   : {SELECTED_ANCHORS_PATH}")

    # This is the important object passed downstream.
    filtered_mentions_rows = dataframe_to_original_row_shape(
        filtered_mentions_df,
        loaded_tables.mentions_rows,
    )

else:
    print("[destructive-filter] Disabled.")
    filtered_mentions_rows = loaded_tables.mentions_rows

[destructive-filter] Building foreign-anchor bank...
[destructive-filter] Selected anchors: 23
[destructive-filter] Done.
  Mentions before : 14,633
  Mentions after  : 13,894
  Dropped         : 739 (5.05%)
  Audit dropped   : c:\Università\Magistrale\NLP - Natural Language Processing\00 Progetto Finale\src_MAVERICK_SpaCyTagger\WORKING DIR\outputs\global_coref\destructive_mention_filter\mentions.dropped.csv
  Audit anchors   : c:\Università\Magistrale\NLP - Natural Language Processing\00 Progetto Finale\src_MAVERICK_SpaCyTagger\WORKING DIR\outputs\global_coref\destructive_mention_filter\selected_anchors.csv


##### Global coreference cluster merging

In [23]:
global_coref_result = merge_local_coreference_clusters(
    clusters_rows=loaded_tables.clusters_rows,
    mentions_rows=filtered_mentions_rows,
    output_dir=GLOBAL_COREF_OUTPUT_DIR,
    verbose=True,
)

print("Final global clusters:", len(global_coref_result.components))

[global-coref] initialized 1623 merge components from 1623 local clusters
[global-coref][exactMention_stitching][step 1] components 1623 -> 1622; candidate_pairs=37; accepted_edges=37; chosen_edge=(merge_component_000808, merge_component_000990); new_component=merge_component_001623; reason=shared normalized canonical name and shared exact overlap mention; metrics={'shared_canonical_count': 1, 'shared_overlap_mention_count': 31}
[global-coref][exactMention_stitching][step 2] components 1622 -> 1621; candidate_pairs=36; accepted_edges=36; chosen_edge=(merge_component_000807, merge_component_000991); new_component=merge_component_001624; reason=shared normalized canonical name and shared exact overlap mention; metrics={'shared_canonical_count': 1, 'shared_overlap_mention_count': 25}
[global-coref][exactMention_stitching][step 3] components 1621 -> 1620; candidate_pairs=35; accepted_edges=35; chosen_edge=(merge_component_000518, merge_component_000688); new_component=merge_component_00162

In [24]:
pd.read_csv(GLOBAL_COREF_OUTPUT_DIR / "global_clusters.csv")

,global_cluster_uid,n_local_clusters,n_mentions,canonical_name,canonical_names,local_cluster_uids,chunk_min,chunk_max
0,global_cluster_000000,1,1,the great Kansas prairies,the great kansas prairies,chunk_000::cluster_000,0,0
1,global_cluster_000001,17,147,Kansas,home to kansas|kansas,chunk_000::cluster_064|chunk_001::cluster_070|...,0,15
2,global_cluster_000002,8,55,Uncle Henry,the tin woodman|uncle henry,chunk_000::cluster_005|chunk_006::cluster_106|...,0,15
3,global_cluster_000003,11,35,a farmer,a farmer|good farmers|the farmer,chunk_000::cluster_001|chunk_000::cluster_003|...,0,15
4,global_cluster_000004,11,126,Aunt Em,aunt em|my aunt em,chunk_000::cluster_002|chunk_002::cluster_067|...,0,14
...,...,...,...,...,...,...,...,...
561,global_cluster_000561,1,1,no one,no one,chunk_015::cluster_025,15,15
562,global_cluster_000562,1,1,the people,the people,chunk_015::cluster_059,15,15
563,global_cluster_000563,1,1,the people,the people,chunk_015::cluster_060,15,15
564,global_cluster_000564,1,1,her loving comrades,her loving comrades,chunk_015::cluster_073,15,15


In [25]:
pd.read_csv(GLOBAL_COREF_OUTPUT_DIR / "global_mentions.csv")

,global_cluster_uid,local_cluster_uid,local_mention_uid,chunk_id,chunk_index,local_cluster_id,local_mention_id,local_start,local_end,global_start,global_end,text,head_local_i,head_global_i,zone,overlap_exact_span_key
0,global_cluster_000000,chunk_000::cluster_000,chunk_000::cluster_000::mention_000,chunk_000,0,0,0,6,10,6,10,the great Kansas prairies,9,9,core,NaN
1,global_cluster_000001,chunk_000::cluster_064,chunk_000::cluster_064::mention_000,chunk_000,0,64,0,8,9,8,9,Kansas,8,8,core,NaN
2,global_cluster_000001,chunk_000::cluster_064,chunk_000::cluster_064::mention_001,chunk_000,0,64,1,2598,2599,2598,2599,Kansas,2598,2598,core,NaN
3,global_cluster_000001,chunk_000::cluster_064,chunk_000::cluster_064::mention_002,chunk_000,0,64,2,2643,2644,2643,2644,Kansas,2643,2643,core,NaN
4,global_cluster_000001,chunk_000::cluster_064,chunk_000::cluster_064::mention_003,chunk_000,0,64,3,2651,2653,2651,2653,that country,2652,2652,core,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13889,global_cluster_000561,chunk_015::cluster_025,chunk_015::cluster_025::mention_000,chunk_015,15,25,0,1136,1138,45757,45759,no one,1137,45758,left_overlap,45757:45759:no one
13890,global_cluster_000562,chunk_015::cluster_059,chunk_015::cluster_059::mention_000,chunk_015,15,59,0,2218,2220,46839,46841,the people,2219,46840,core,NaN
13891,global_cluster_000563,chunk_015::cluster_060,chunk_015::cluster_060::mention_000,chunk_015,15,60,0,2280,2282,46901,46903,the people,2281,46902,core,NaN
13892,global_cluster_000564,chunk_015::cluster_073,chunk_015::cluster_073::mention_000,chunk_015,15,73,0,3066,3069,47687,47690,her loving comrades,3068,47689,core,NaN
